# Exploratory Data Analysis (EDA)

 is the process of investigating a dataset to summarize its main characteristics — often using visuals and statistics — before you clean it, model it, or trust any conclusion from it.

**Why it matters:**

You cannot preprocess data correctly if you don't understand it first (garbage decisions in → garbage pipeline out).
It reveals data quality issues: missing values, wrong types, outliers, duplicates, inconsistent categories.
It reveals the story in the data: trends, relationships, imbalances — which drives feature engineering and modeling choices.
It prevents costly mistakes, like training a model on a leaked or broken column.

Rule: Never preprocess or model a dataset you haven't explored. EDA always comes before cleaning decisions, because cleaning decisions depend on what EDA finds.

# The EDA Mindset: Questions to Ask

Before writing any code, form the habit of asking:

What does each row represent? (a person? a transaction? a day?)
What does each column mean? Do I need a data dictionary?
What is the size of the data — rows, columns, memory?
What are the data types, and do they make sense?
Are there missing values? Where, and how much?
Are there duplicates?
Are there impossible or suspicious values (negative age, future dates, 0 income for a "salary" column)?
What does the distribution of each important variable look like?
Are there relationships between variables that matter for my goal?
Is the data balanced (for categories/classes) or skewed?

EDA is answering these questions systematically, with code.

**Step 1 — First Look at the Data**

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data.csv')

df.head()      # first 5 rows
df.tail()       # last 5 rows
df.sample(5)     # random 5 rows -- often more revealing than head()

**Step 2 — Structural Understanding**

Understand the "shape" of your dataset before anything else.

In [ ]:
df.shape              # (rows, columns)
df.columns.tolist()    # column names
df.dtypes              # data type of each column
df.info()              # combines dtype + non-null counts + memory usage

In [ ]:
df.info()

From this alone you already know: age and income have missing values, and join_date should probably be converted from object to datetime.

**Step 3 — Univariate Analysis (One Column at a Time)**

**Numeric columns**

In [ ]:
df.describe()

This gives: count, mean, std, min, 25%, 50% (median), 75%, max.

What to look for:

min/max — are they plausible? (age = -5 or age = 300 is a red flag)

mean vs median (50%) — if very different, the distribution is skewed

std — how spread out is the data?
Big jump between 75% and max → possible outliers

In [ ]:
df['age'].describe()
df['age'].hist(bins=30)
df['age'].plot(kind='box')     # boxplot for outliers

**Categorical / text columns**

In [ ]:
df['city'].value_counts()
df['city'].value_counts(normalize=True) * 100   # as percentages
df['city'].nunique()                              # number of unique categories

***What to look for:***

Are categories consistent? ("Karachi", "karachi", "KARACHI " should be one category, not three)
Is one category dominating (class imbalance)?
Are there rare categories that might need grouping into "Other"?

**Step 4 — Missing Data Analysis**

In [ ]:
df.isnull().sum()
(df.isnull().sum() / len(df) * 100).round(2)   # % missing per column

Ask:

Is data missing completely at random, or does it follow a pattern?

 (e.g., income is missing only for unemployed people — that's meaningful, not random)
Which columns have so much missing data that they may need to be dropped?

Does "missing" itself carry information? (Sometimes NaN should become a category like "Not Provided" rather than be imputed.)

**Step 5 — Duplicate & Consistency Checks**

In [ ]:
df.duplicated().sum()                     # exact duplicate rows
df.duplicated(subset=['id']).sum()          # duplicate keys (should usually be 0)

df['city'].unique()          # scan for inconsistent spellings/casing
df['email'].str.contains('@').value_counts()   # sanity-check format

Consistency red flags to check:
**bold text**
Same entity spelled differently ("USA" vs "United States")

Whitespace issues (" Lahore" vs "Lahore")

Mixed units (some prices in USD, some in PKR)

Inconsistent date formats

**Step 6 — Outlier Detection**

*Outliers can be genuine (a billionaire in an income dataset) or data errors (age = 999). EDA is where you decide which.*

In [ ]:
# IQR method
Q1 = df['income'].quantile(0.25)
Q3 = df['income'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['income'] < lower) | (df['income'] > upper)]

# Z-score method
from scipy import stats
z_scores = stats.zscore(df['income'].dropna())
outliers_z = df.loc[df['income'].dropna().index][abs(z_scores) > 3]

# Visual
df.boxplot(column='income')

**Step 7 — Categorical Data Analysis**

In [ ]:
df['category'].value_counts().plot(kind='bar')

# Check class imbalance for a target variable
df['target'].value_counts(normalize=True)

If one class is 95% and another is 5%, that's a class imbalance — important for modeling decisions (affects metric choice and resampling strategy).